In [1]:
%%capture
%load_ext autoreload
%autoreload 2

In [2]:
import os

import warnings
from pprint import pprint
warnings.filterwarnings('ignore')
os.chdir("/Users/azrrael/Eafit/ui-signal-plotter")

In [3]:
from src.domain.entities.component import Component, Value, Range
from src.domain.value_objects import DeviceType, Unit

In [4]:
fan = Component(
    id=0,
    name="FAN",
    description="Fan controller",
    type=DeviceType.ACTUATOR,
    unit=Unit.RPM,
    range=Range(min=0, max=100, unit=Unit.RPM)
)

In [7]:
fan.get_average()

0

In [17]:
from src.application.services import get_component_status_service, add_values_service, get_component_values_service, forecast_value_service

In [8]:
get_component_status_service(fan)

<DeviceStatus.OFFLINE: 'offline'>

In [11]:
from random import randint

In [14]:
add_values_service(
    fan, [Value(value=randint(0, 100), unit=Unit.RPM) for _ in range(10)]
)

10

In [15]:
get_component_values_service(fan)

[Value(value=54, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=28, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=41, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=45, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=57, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=60, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=0, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=67, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=93, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=25, unit=<Unit.RPM: 'RPM'>, timestamp=1745682506782),
 Value(value=13, unit=<Unit.RPM: 'RPM'>, timestamp=1745682596187),
 Value(value=16, unit=<Unit.RPM: 'RPM'>, timestamp=1745682596187),
 Value(value=23, unit=<Unit.RPM: 'RPM'>, timestamp=1745682596187),
 Value(value=10, unit=<Unit.RPM: 'RPM'>, timestamp=1745682596187),
 Value(value=69, unit=<Unit.RPM: 'RPM'>, timestamp=174568259618

In [18]:
forecast_value_service(fan)

Value(value=np.float64(33.00526315789473), unit=<Unit.RPM: 'RPM'>, timestamp=1745682620947)

In [28]:
from src.infrastructure.repositories.memory import MemoryComponentRepository

repository = MemoryComponentRepository()

In [29]:
from src.application.use_cases import (
    create_component_use_case,
    ComponentDTO,
    RangeDTO,
    add_value_to_component_use_case,
    get_component_values_use_case,
    delete_component_use_case,
)

component = create_component_use_case(
    component=ComponentDTO(
        name="FAN",
        description="Fan controller",
        type="actuator",
        unit="RPM",
        range=RangeDTO(min=0, max=100, unit="RPM"),
    ),
    repository=repository,
)

component = create_component_use_case(
    component=ComponentDTO(
        name="LM35",
        description="Temperature sensor",
        type="sensor",
        unit="C",
        range=RangeDTO(min=-273, max=100, unit="C"),
    ),
    repository=repository,
)

In [30]:
values = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1000]

In [31]:
updates = add_value_to_component_use_case(
    component_id=0,
    values=values,
    repository=repository
)


Invalid value: Value(value=1000, unit=<Unit.RPM: 'RPM'>, timestamp=1745683270309)


In [32]:
reads = get_component_values_use_case(
    component_id=0,
    repository=repository
)

reads

[{'value': 10, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 20, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 30, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 40, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 50, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 60, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 70, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 80, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 90, 'unit': 'RPM', 'timestamp': 1745683270309},
 {'value': 100, 'unit': 'RPM', 'timestamp': 1745683270309}]

In [42]:
args = {
    "component_id": 1,
    "repository": repository
}

In [43]:
reads = get_component_values_use_case(**args)

In [44]:
reads

[]

In [8]:
delete_component_use_case(
    component_id=1,
    repository=repository
)


In [34]:
import requests

In [35]:
payload = {
    "name": "FAN",
    "description": "Fan controller",
    "type": "actuator",
    "unit": "RPM",
    "range": {
        "min": 0,
        "max": 100,
        "unit": "RPM"
    }
}

response = requests.post(
    "http://localhost:8000/components", json=payload
)

In [36]:
response.json()

{'id': 0,
 'name': 'FAN',
 'description': 'Fan controller',
 'type': 'actuator',
 'unit': 'RPM',
 'range': {'min': 0.0, 'max': 100.0, 'unit': 'RPM'},
 'values': [],
 'status': 'offline'}

In [39]:
values = {"values": [randint(0, 100) for _ in range(10)]}

response = requests.put(
    "http://localhost:8000/components/0/values", json=values
)

response.json()

{'updates': 10}

In [41]:
response = requests.get(
    "http://localhost:8000/components/0/values"
)

response.json()



[{'value': 87.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 22.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 62.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 0.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 27.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 11.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 61.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 15.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 96.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 65.0, 'unit': 'RPM', 'timestamp': 1745683509828},
 {'value': 63.0, 'unit': 'RPM', 'timestamp': 1745684731610},
 {'value': 93.0, 'unit': 'RPM', 'timestamp': 1745684731610},
 {'value': 18.0, 'unit': 'RPM', 'timestamp': 1745684731610},
 {'value': 23.0, 'unit': 'RPM', 'timestamp': 1745684731610},
 {'value': 59.0, 'unit': 'RPM', 'timestamp': 1745684731610},
 {'value': 55.0, 'unit': 'RPM', 'timestamp': 1745684731610},
 {'value': 96.0, 'unit': 

In [14]:
# response = requests.delete(
#     "http://localhost:8000/components/0"
# )

# response.json()
